In [0]:
%python
# Databricks notebook source
# PURPOSE:
# 1) Build chunk table for retrieval
# 2) Generate embeddings (placeholder UDF; replace with your embedding endpoint call)
# 3) Save vector-ready table

from pyspark.sql import functions as F

CATALOG = "vf_health"
SCHEMA = "ghana"

PROFILES = f"{CATALOG}.{SCHEMA}.gold_facility_profiles"
CLAIMS = f"{CATALOG}.{SCHEMA}.gold_facility_claims"

CHUNKS = f"{CATALOG}.{SCHEMA}.gold_profile_chunks"

profiles_df = spark.table(PROFILES).select("row_id", "unique_id", "name", "profile_json")
claims_df = spark.table(CLAIMS).groupBy("row_id").agg(
    F.collect_list("claim_text").alias("claim_texts")
)

joined = (
    profiles_df.join(claims_df, on="row_id", how="left")
    .withColumn("claim_text_joined", F.array_join(F.coalesce("claim_texts", F.array()), " | "))
    .withColumn(
        "chunk_text",
        F.concat_ws(
            " || ",
            F.coalesce("name", F.lit("")),
            F.coalesce("profile_json", F.lit("")),
            F.coalesce("claim_text_joined", F.lit(""))
        )
    )
    .select("row_id", "unique_id", "name", "chunk_text")
)

# Placeholder embedding vector (replace with real embedding endpoint output)
# For now create deterministic pseudo vector dimensions for pipeline continuity
joined = joined.withColumn(
    "embedding",
    F.array(
        F.length("chunk_text").cast("double"),
        (F.length("chunk_text") % 97).cast("double"),
        (F.length("chunk_text") % 53).cast("double")
    )
)

joined.write.mode("overwrite").format("delta").saveAsTable(CHUNKS)

print("Chunks table rows:", spark.table(CHUNKS).count())
display(spark.table(CHUNKS).limit(10))

# NOTE:
# Next in UI/SDK create Databricks Vector Search index on:
# table: vf_health.ghana.gold_profile_chunks
# PK: row_id
# text column: chunk_text
# embedding column: embedding (or model-generated column if using managed embeddings)